In [1]:
## **RNN(RECURRENT NEURAL N/W)** ##

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,SimpleRNN,Dense

c:\Users\MY PC\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
sentences = [
    # Positive (15)
    "I love this product",
    "This movie made me smile",
    "Service was friendly and quick",
    "Today felt bright and happy",
    "This is the best day",
    "Absolutely fantastic experience",
    "I enjoyed every single moment",
    "Great job, well done",
    "The food tasted delicious",
    "Totally recommend to everyone",
    "Very satisfied with results",
    "This worked better than expected",
    "Amazing quality and value",
    "Such a pleasant surprise",
    "I feel positive about this",

    # Negative (15)
    "I hate this product",
    "This movie was terrible",
    "Service was very slow",
    "Today was the worst day",
    "I am extremely disappointed",
    "The food tasted awful",
    "This is a complete waste of money",
    "Very poor quality",
    "I will never buy this again",
    "The experience was horrible",
    "Everything went wrong",
    "I regret buying this",
    "The product stopped working",
    "Customer support was useless",
    "Not worth the price"
]

labels = [1] * 15 + [0] * 15
labels = np.array(labels)

In [4]:
vocab_size = 2000

# Create tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
#Tokenizer har unique word ko ek index (number) de deta hai based on word frequency.
# Learn vocabulary from the sentences
tokenizer.fit_on_texts(sentences)

# Convert sentences into integer sequences
seqs = tokenizer.texts_to_sequences(sentences)

# Find the length of the longest sequence
maxlen = max(len(s) for s in seqs)

# Pad all sequences to the same length
padded = pad_sequences(seqs, maxlen=maxlen, padding='post')#post -words complete ho jae then add padding 
y=labels
print(padded)

[[ 3 19  2  6  0  0  0]
 [ 2  9 20 21 22  0  0]
 [10  5 23  7 24  0  0]
 [11 25 26  7 27  0  0]
 [ 2 12  4 28 13  0  0]
 [29 30 14  0  0  0  0]
 [ 3 31 32 33 34  0  0]
 [35 36 37 38  0  0  0]
 [ 4 15 16 39  0  0  0]
 [40 41 42 43  0  0  0]
 [ 8 44 45 46  0  0  0]
 [ 2 47 48 49 50  0  0]
 [51 17  7 52  0  0  0]
 [53 18 54 55  0  0  0]
 [ 3 56 57 58  2  0  0]
 [ 3 59  2  6  0  0  0]
 [ 2  9  5 60  0  0  0]
 [10  5  8 61  0  0  0]
 [11  5  4 62 13  0  0]
 [ 3 63 64 65  0  0  0]
 [ 4 15 16 66  0  0  0]
 [ 2 12 18 67 68 69 70]
 [ 8 71 17  0  0  0  0]
 [ 3 72 73 74  2 75  0]
 [ 4 14  5 76  0  0  0]
 [77 78 79  0  0  0  0]
 [ 3 80 81  2  0  0  0]
 [ 4  6 82 83  0  0  0]
 [84 85  5 86  0  0  0]
 [87 88  4 89  0  0  0]]


In [5]:
maxlen

7

 The Embedding layer is used because neural networks cannot understand word IDs directly. It converts each word into a meaningful numerical representation that captures relationships between words.

 I love this product

 [3, 19, 2, 6]

 Here:

3 is just the ID for I
19 is just the ID for love
2 is just the ID for this
6 is just the ID for product

The problem is that these numbers do not have any meaning.

For example:
love  = 19
hate  = 20
apple = 50

Does 20 mean "hate" is bigger than "love"? ❌ No.

These are only labels (IDs).

The Embedding layer converts each word ID into a vector


 [0.12, 0.45, -0.21, ..., 0.33],   # I
 [0.87, 0.65,  0.91, ..., 0.10],   # love
 [0.31, 0.22,  0.18, ..., 0.72],   # this
 [0.44, 0.91, -0.50, ..., 0.28]    # product
]

Why is this useful?

During training, the model learns similar vectors for words with similar meanings.

In [6]:
embed_dim=16
rnn_units=8

rnn_units specifies how many neurons (hidden units) the RNN layer has. These neurons act as the RNN's memory while reading a sequence.
means the RNN has 8 hidden units.

In [7]:
inp = Input(shape=(maxlen,), dtype='int32', name='input')

x = Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim,   # No quotes ✅
    mask_zero=True,
    name='embed'
)(inp)

* **`embed_dim = 16`** → Defines that each word will be represented by a vector containing **16 numerical features**.

* **`rnn_units = 8`** → Specifies that the RNN will use **8 hidden neurons (memory units)** to learn and remember patterns in the sequence.

* **Input Layer** → Accepts the padded sequence of word IDs as the input to the model.

* **`input_dim = vocab_size`** → Tells the embedding layer the total number of unique words in the vocabulary.

* **`output_dim = embed_dim`** → Converts each word ID into an embedding vector of size **16**.

* **`mask_zero = True`** → Instructs the model to ignore the padding value (`0`) while processing the sequence.

* **`name = 'embed'`** → Assigns the name **"embed"** to the embedding layer for easy identification.

* **`(inp)`** → Connects the input layer to the embedding layer so that the input word IDs are transformed into embedding vectors before being passed to the RNN.


rnn = SimpleRNN(units=rnn_units, return_sequences=True, return_state=True, name='simple_rnn')#

In [8]:
rnn = SimpleRNN(units=rnn_units, return_sequences=True, return_state=True, name='simple_rnn')
#Creates a SimpleRNN layer with rnn_units neurons that returns both the full output sequence and the final hidden state.

output, state = rnn(x)
#Passes the embedded word vectors through the RNN to learn sequential patterns.

out = Dense(1, activation='sigmoid', name='out')(state)
#Uses a sigmoid activation to predict the probability of the input belonging to the positive class.

model = Model(inputs=inp, outputs=out)
#Builds the complete neural network by connecting the input layer to the output layer.

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
#Configures the model using the Adam optimizer, binary cross-entropy loss, and accuracy as the evaluation metric.

model.summary()
#Displays the architecture of the model, including each layer, output shape, and number of trainable parameters.

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 7)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 7, 16)     │     32,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 7)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 7, 8),    │        200 │ embed[0][0],      │
│ (SimpleRNN)         │ (None, 8)]        │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[0][1]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
print(type(x))
print(type(y))

print(x.shape)
print(y.shape)

<class 'keras.src.backend.common.keras_tensor.KerasTensor'>
<class 'numpy.ndarray'>
(None, 7, 16)
(30,)


In [10]:
model.fit(padded,labels,epochs=25,batch_size=8,verbose=1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.4333 - loss: 0.7031
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6667 - loss: 0.6834
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8000 - loss: 0.6653
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8333 - loss: 0.6468
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9000 - loss: 0.6287
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9667 - loss: 0.6115
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.9667 - loss: 0.5927
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9667 - loss: 0.5715
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9667 - loss: 0.5511
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.9333 - loss: 0.5295
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.9333 - loss: 0.5071
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9333 - loss: 0.4852
E

In [11]:
test = ["i do not love this movie"]

seq = tokenizer.texts_to_sequences(test)
pad = pad_sequences(seq, maxlen=maxlen, padding='post')

prediction = model.predict(pad)
print(prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 351ms/step
[[0.40647867]]


In [12]:
prediction = model.predict(pad)

if prediction[0][0] >= 0.5:
    print("Positive 😊")
else:
    print("Negative 😞")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
Negative 😞


                          START
                             │
                             ▼
                  Create Sentences & Labels
                             │
                             ▼
         Positive Sentences → Label = 1
         Negative Sentences → Label = 0
                             │
                             ▼
                     Tokenizer Created
                             │
                             ▼
              tokenizer.fit_on_texts(sentences)
                             │
                             ▼
          Learns Vocabulary & Assigns Word IDs
                             │
                             ▼
          tokenizer.texts_to_sequences(sentences)
                             │
                             ▼
      "I love this product"
                 │
                 ▼
           [3, 19, 2, 6]
                             │
                             ▼
                 pad_sequences()
                             │
                             ▼
         [3, 19, 2, 6, 0, 0, 0]
          (All sentences become equal length)
                             │
                             ▼
                     Input Layer
        (Accepts padded integer sequences)
                             │
                             ▼
                  Embedding Layer
          (Each word ID → 16-dimensional vector)
                             │
                             ▼
                 Example Transformation

        [3, 19, 2, 6, 0, 0, 0]

                    │
                    ▼

   [
     [16 values],   ← I
     [16 values],   ← love
     [16 values],   ← this
     [16 values],   ← product
     Padding ignored (mask_zero=True)
   ]
                             │
                             ▼
                  SimpleRNN Layer
          (Reads one word at a time and
         remembers information using 8 units)
                             │
                             ▼
                 Final Hidden State
                 (Summary of sentence)
                             │
                             ▼
                     Dense Layer
          (1 Neuron + Sigmoid Activation)
                             │
                             ▼
                 Output Probability
                     Example: 0.7303
                             │
                 ┌───────────┴───────────┐
                 │                       │
        Probability ≥ 0.5        Probability < 0.5
                 │                       │
                 ▼                       ▼
        Positive Sentiment        Negative Sentiment
            (Label = 1)             (Label = 0)
                             │
                             ▼
                         END

In [13]:
import sys
print(sys.executable)

c:\Users\MY PC\anaconda3\python.exe


In [ ]:
# Save the model
model.save("model.keras")

# Save the tokenizer
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)